In [1]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = True
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [2]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

import random
import math

import pandas as pd

import copy

import cvxpy
cp = cvxpy

import figurefirst as fifi

from braid_analysis import braid_analysis_plots
from braid_analysis import flymath
import fly_plot_lib.plot as fpl

import pynumdiff

from sklearn.preprocessing import MinMaxScaler

import scipy.stats

In [3]:
import sys
from pathlib import Path


In [4]:
# import sys
# sys.path.append('../notebooks')
# from splitflow import unifying_algo_analysis_helper as uaah

# Helper functions

In [5]:
from splitflow.affine_math_and_plot_helper import *

# Hyper parameters

In [6]:
RANDOMNESS = False

FIGURE_NAME = 'unifying_math_v4.svg'

# 
SPEED = 0.3

UPWIND_GAMMA_MULTIPLIER = 0

UPWIND_TRANSLATION_MULTIPLIER = 0.2

SACCADE_THRESHOLD_PROPORTIONAL = 8
SACCADE_THRESHOLD_DERIVATIVE = 3

SMOOTHING_WINDOW = 11

SLOPE_STD = 0.3

dt = 0.01

# Rerun this notebook for all four options:

stillair, laminar, unsteady, low

In [7]:
scenario = 'unsteady'

t, wind_direction, wind_speed = get_wind(scenario, RANDOMNESS=True, SMOOTHING_WINDOW=SMOOTHING_WINDOW, dt=dt)
raw_circle_course = get_circle_basis(t, RANDOMNESS)
axis_ratio = get_axis_ratio(wind_direction, wind_speed)
color = get_color(scenario)

In [8]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

### Plot the wind time series

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from typing import List, Optional, Tuple


def circular_distribution_plot(
    directions: List[float],
    magnitudes: List[float],
    ax: Optional[plt.Axes] = None,
    bins: int = 36,
    radial_limit: Optional[float] = None,
    log_scale: bool = False,
    normalize: bool = False,
    figsize: Tuple[int, int] = (7, 7),
) -> plt.Axes:
    """
    Create a circular distribution (polar) plot for vector directions and magnitudes.

    Bar height encodes mean magnitude per bin. Bar darkness encodes count density
    (how many vectors fall in that bin), mapped from light gray (few) to black (many).
    Together these show both the strength and prevalence of vectors in each direction.

    Parameters
    ----------
    directions   : Sequence of direction angles in radians (-pi to pi).
    magnitudes   : Corresponding magnitudes for each direction vector.
    ax           : Existing polar Axes to draw into. If None, a new figure is created.
    bins         : Number of angular bins around the circle (default 36 -> 10 deg each).
    radial_limit : Upper limit of the radial axis. Applied after any transforms.
    log_scale    : If True, apply log1p transform to mean magnitudes before plotting.
    normalize    : If True, normalize mean magnitudes to sum to 1 across bins.
                   Useful for comparing distributions with different overall scales.
    figsize      : Figure size in inches, used only when ax is None.

    Returns
    -------
    ax           : The polar Axes object.

    Example
    -------
    >>> import numpy as np
    >>> rng = np.random.default_rng(42)
    >>> dirs = rng.uniform(-np.pi, np.pi, 200)
    >>> mags = rng.rayleigh(scale=5, size=200)
    >>> ax = circular_distribution_plot(dirs, mags)
    """
    directions = np.asarray(directions, dtype=float)
    magnitudes = np.asarray(magnitudes, dtype=float)

    if directions.shape != magnitudes.shape:
        raise ValueError("directions and magnitudes must have the same length.")

    # --- Binning ---
    radians = directions % (2 * np.pi)
    bin_edges = np.linspace(0, 2 * np.pi, bins + 1)
    bin_width = bin_edges[1] - bin_edges[0]

    bin_indices = np.clip(np.digitize(radians, bin_edges) - 1, 0, bins - 1)

    bin_total_mag = np.zeros(bins)
    bin_counts = np.zeros(bins, dtype=float)
    for idx, mag in zip(bin_indices, magnitudes):
        bin_total_mag[idx] += mag
        bin_counts[idx] += 1

    # Mean magnitude per bin (bar height); empty bins stay at 0
    bin_mean_mag = np.where(bin_counts > 0, bin_total_mag / bin_counts, 0.0)

    # --- Transforms ---
    if normalize and bin_mean_mag.sum() > 0:
        bin_mean_mag = bin_mean_mag / bin_mean_mag.sum()

    if log_scale:
        bin_mean_mag = np.log1p(bin_mean_mag)

    # --- Bar colours: count density mapped to gray (light = few, dark = many) ---
    count_norm = Normalize(vmin=0, vmax=bin_counts.max() if bin_counts.max() > 0 else 1)
    # Map normalised count to a gray value: 0 -> 0.85 (light), 1 -> 0.15 (dark)
    gray_values = 0.85 - 0.70 * count_norm(bin_counts)
    bar_colors = [(g, g, g, 1.0) for g in gray_values]

    # --- Axes ---
    if ax is None:
        _, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": "polar"})

    ax.set_theta_zero_location("E")
    ax.set_theta_direction(1)

    # --- Bars ---
    bars = ax.bar(
        bin_edges[:-1],
        bin_mean_mag,
        width=bin_width,
        color=bar_colors,
        edgecolor="none",
        linewidth=0.4,
        align="edge",
    )

    # --- Clean up labels and ticks ---
    ax.set_xticklabels([])
    #ax.set_yticklabels([])
    ax.grid(color="grey", linestyle="--", linewidth=0.5, alpha=0.4)

    if radial_limit is not None:
        ax.set_ylim(0, radial_limit)

    ax.spines["polar"].set_visible(False)
    
    return ax


# ---------------------------------------------------------------------------
# Quick demo
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    rng = np.random.default_rng(0)

    # Cluster 1: many vectors, high magnitude (NE)
    # Cluster 2: few vectors, low magnitude (SW)
    # Background: sparse uniform noise
    dirs = np.concatenate([
        rng.normal(loc=np.pi/4,  scale=0.35, size=300),
        rng.normal(loc=-2.8,     scale=0.5,  size=30),
        rng.uniform(-np.pi, np.pi,           size=50),
    ])
    mags = np.concatenate([
        rng.rayleigh(scale=8,  size=300),
        rng.rayleigh(scale=12, size=30),
        rng.rayleigh(scale=2,  size=50),
    ])

    ax = circular_distribution_plot(dirs, 
                                    mags, 
                                    normalize=False) #, radial_limit=0.1)
    plt.show()

/var/folders/63/jqdzyjdd4bnghtd7s779bx500000gp/T/ipykernel_9584/3076420186.py:69: RuntimeWarning: invalid value encountered in divide
  bin_mean_mag = np.where(bin_counts > 0, bin_total_mag / bin_counts, 0.0)


RuntimeError: latex was not able to process the following string:
b'lp'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) (preloaded format=latex)
 restricted \write18 enabled.

kpathsea: Running mktexfmt latex.fmt
Can't locate mktexlsr.pl in @INC (@INC contains: /opt/anaconda3/envs/Python311/share/tlpkg /opt/anaconda3/envs/Python311/share/texmf-dist/scripts/texlive /Library/Perl/5.34/darwin-thread-multi-2level /Library/Perl/5.34 /Network/Library/Perl/5.34/darwin-thread-multi-2level /Network/Library/Perl/5.34 /Library/Perl/Updates/5.34.1 /System/Library/Perl/5.34/darwin-thread-multi-2level /System/Library/Perl/5.34 /System/Library/Perl/Extras/5.34/darwin-thread-multi-2level /System/Library/Perl/Extras/5.34) at /opt/anaconda3/envs/Python311/bin/mktexfmt line 41.
BEGIN failed--compilation aborted at /opt/anaconda3/envs/Python311/bin/mktexfmt line 43.
I can't find t

<Figure size 700x700 with 1 Axes>

In [10]:
if 0: 
    ax = layout.axes[(scenario, 'wind')]
    
    #####################################################################################################
    # Wind
    wind_direction_shifted = wrap_angle(wind_direction, midangle='pi')+np.pi
    ax.scatter(t, wind_direction_shifted, s=2, c=wind_speed, cmap='plasma', vmin=0, vmax=0.6, zorder=-1)
    
    ax.set_ylim(0, 2*np.pi)
    
    xticks = [0, 1, 2, 3, 4, 5]
    xticklabels = np.array(xticks)
    ax.set_xlim(0, 5)
    ax.set_xticks(xticks)
    ax.set_xticklabels([])
    
    ax.set_yticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
    
    if scenario == 'stillair':
        ax.set_yticklabels(['$0$', '', '', '', '$2\pi$'])
        ax.tick_params(axis='y', pad=2)
        ax.tick_params(axis='x', pad=2)
    else:
        ax.set_yticklabels([])
    
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)
    
    ax.set_rasterization_zorder(0)

if 1:
    #fig = plt.figure()
    #ax = fig.add_subplot(111, projection='polar')
    ax = layout.axes[(scenario, 'wind')]
    
    circular_distribution_plot(wind_direction,
                               wind_speed,
                                    ax = ax,
                                    bins = 50,
                                    normalize=False,
                                    #radial_limit=0.1
                                )

    if scenario == 'low':
        ax.set_yticks([0, 0.2])
        ax.set_thetagrids([])
        ax.set_ylim(0, 0.5)
        ax.set_yticklabels(['', '0.2 m/s'])
    if scenario == 'laminar':
        #ax.set_yticks([0, 0.2])
        #ax.set_ylim(0, 0.5)
        #ax.set_yticklabels(['', '0.2 m/s'])
        ax.set_yticks([0, 0.2])
        ax.set_thetagrids([])
        ax.set_ylim(0, 0.5)
        ax.set_yticklabels(['', '0.2 m/s'])
    if scenario == 'unsteady':
        ax.set_yticks([0, 0.2])
        ax.set_thetagrids([])
        ax.set_ylim(0, 0.5)
        ax.set_yticklabels(['', '0.2 m/s'])
    if scenario == 'stillair':
        ax.set_yticks([0, 0.02])
        ax.set_thetagrids([])
        ax.set_ylim(0, 0.04)
        ax.set_yticklabels(['', '0.02 m/s'])

    ax.tick_params(axis='y', labelcolor='gray')

/var/folders/63/jqdzyjdd4bnghtd7s779bx500000gp/T/ipykernel_9584/3076420186.py:69: RuntimeWarning: invalid value encountered in divide
  bin_mean_mag = np.where(bin_counts > 0, bin_total_mag / bin_counts, 0.0)


### Get affine warped course

In [11]:
raw_circle_xy = get_circle_xy(raw_circle_course)

affine_trans_warped_course_angle, affine_trans_warped_circle_xy, mean_affine_trans_warped_circle_xy = get_affine_warped_course_angle(axis_ratio, 
                                                                                                                   wind_direction, 
                                                                                                                   raw_circle_xy,
                                                                                                                   translation_magnitude=UPWIND_TRANSLATION_MULTIPLIER*(1-axis_ratio)
                                                                                                                  )



In [12]:
affine_warped_course_angle, affine_warped_circle_xy, mean_affine_warped_circle_xy = get_affine_warped_course_angle(axis_ratio, 
                                                                                                                   wind_direction, 
                                                                                                                   raw_circle_xy,
                                                                                                                  )



### Plot circle and affine transform

In [13]:
if 0:
    ax = layout.axes[(scenario, 'ellipse')]
    ax.set_aspect('equal')
    
    # plot raw circle
    ax.plot(raw_circle_xy[0,0:160], raw_circle_xy[1,0:160], '--', color='blue', linewidth=0.5)
    
    # Plot the affine transform
    #ax.plot(affine_warped_circle_xy[0,:], affine_warped_circle_xy[1,:], color=color, linewidth=0.5)
    ax.plot(mean_affine_trans_warped_circle_xy[0,:], mean_affine_trans_warped_circle_xy[1,:], color=color, linewidth=1)
    
    ax.set_ylim(-1.1, 1.1)
    fifi.mpl_functions.adjust_spines(ax, [])
    ax.set_rasterization_zorder(0)

In [14]:
# cvx_warped_course_angle, cvx_warped_circle_xy = get_upwind_biased_trajec_cvx(wind_direction, axis_ratio, affine_warped_course_angle, 
#                                                                              UPWIND_GAMMA_MULTIPLIER,
#                                                                              SMOOTHING_WINDOW, SPEED, RANDOMNESS=False)
# x, y = get_trajec(cvx_warped_course_angle, dt, SPEED, RANDOMNESS=False)

### Plot upwind biased trajec

In [15]:
x, y = get_trajec(affine_trans_warped_course_angle, dt, SPEED)

In [16]:
#####################################################################################################
# Goal C
ax = layout.axes[(scenario, 'trajec1')]
ax.set_aspect('equal')

if scenario == 'laminar':
    color = '#991128ff'
if scenario == 'unsteady':
    color = '#2b75b3ff'
if scenario == 'still':
    color = '#084a72ff'
if scenario == 'low':
    color = '#bd7bffff'

orig_traj = np.vstack((x, y)).T
orig_traj = MinMaxScaler().fit_transform(orig_traj)
braid_analysis_plots.plot_arrowhead_trajectory(orig_traj[:,0]*0.98, 
                                               orig_traj[:,1]*0.98, 
                                               color=color, 
                                               linewidth=0.75, 
                                               ax=ax, arrow_length=0.15)

fifi.mpl_functions.adjust_spines(ax, [])

### Plot upwind biased cvx course

In [17]:
if 0:
    ax = layout.axes[(scenario, 'course1')]
    plot_course(ax, t, raw_circle_course, affine_warped_course_angle, affine_trans_warped_course_angle, 
                colors=['blue', 'none', 'magenta'],
                    show_labels=False)
    set_selective_rasterization(ax, 
                                rasterize_markers=['.'], 
                                rasterize_collections=[mcollections.PathCollection, 
                                                       mcollections.PolyCollection], 
                                raster_zorder=-1)
    
    if scenario == 'stillair':
        ax.set_yticklabels(['$-\pi$', '', '', '', '$\pi$'])
        ax.set_xticklabels(['$0$', '', '', '', '', '$5$'])

### Get upwind bias with saccadic trajectory

In [18]:
staircase_warped_course_angle = get_upwind_biased_trajec_saccade(wind_direction, axis_ratio, affine_warped_circle_xy, 
                                                                 dt,
                                                                 UPWIND_GAMMA_MULTIPLIER,
                                                                 SMOOTHING_WINDOW, SPEED, 
                                                                 SACCADE_THRESHOLD_PROPORTIONAL, SACCADE_THRESHOLD_DERIVATIVE,
                                                                 )

In [19]:
x, y = get_trajec(staircase_warped_course_angle, dt, SPEED)

### Plot the saccadic trajectory

In [20]:
# ax = layout.axes[(scenario, 'trajec2')]
# ax.set_aspect('equal')

# orig_traj = np.vstack((x, y)).T
# orig_traj = MinMaxScaler().fit_transform(orig_traj)
# braid_analysis_plots.plot_arrowhead_trajectory(orig_traj[:,0], orig_traj[:,1], color='black', linewidth=0.5, ax=ax, arrow_length=0.12)

# fifi.mpl_functions.adjust_spines(ax, [])

### Plot the saccadic course

In [21]:
# ax = layout.axes[(scenario, 'course2')]
# plot_course(ax, t, raw_circle_course, affine_warped_course_angle, staircase_warped_course_angle, 
#                 show_labels=False)

### Write to SVG

In [22]:
layout.append_figure_to_layer(layout.figures[scenario], scenario, cleartarget=True)
layout.write_svg(FIGURE_NAME)

RuntimeError: latex was not able to process the following string:
b'lp'

Here is the full command invocation and its output:

latex -interaction=nonstopmode --halt-on-error file.tex

This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) (preloaded format=latex)
 restricted \write18 enabled.

kpathsea: Running mktexfmt latex.fmt
Can't locate mktexlsr.pl in @INC (@INC contains: /opt/anaconda3/envs/Python311/share/tlpkg /opt/anaconda3/envs/Python311/share/texmf-dist/scripts/texlive /Library/Perl/5.34/darwin-thread-multi-2level /Library/Perl/5.34 /Network/Library/Perl/5.34/darwin-thread-multi-2level /Network/Library/Perl/5.34 /Library/Perl/Updates/5.34.1 /System/Library/Perl/5.34/darwin-thread-multi-2level /System/Library/Perl/5.34 /System/Library/Perl/Extras/5.34/darwin-thread-multi-2level /System/Library/Perl/Extras/5.34) at /opt/anaconda3/envs/Python311/bin/mktexfmt line 41.
BEGIN failed--compilation aborted at /opt/anaconda3/envs/Python311/bin/mktexfmt line 43.
I can't find t

### Display SVG

In [ ]:
from IPython.display import display,SVG
display(SVG(FIGURE_NAME))